# Análisis Estadístico de Rendimiento - SGROAS
**Bloque C.1** | Fuente: `docs/mediciones/perf/` | Método: `scripts/perf-analysis.py` (`scripts/perf/k6_loader.py` lee `metrics["http_req_duration"]`)
Serie K1 (local, n=3: k01..k03, 50 VUs, 30 s, `GET /api/conductores` con JWT) + serie Render (n=5 calientes K4..K8 + 5 muestras frías, commit `9ded2a69`).
Al abrir este cuaderno se ve el último run que produjo los números del documento (D.1). Rutas relativas a la raíz del repo.

In [1]:
import json, pathlib
est = json.loads(pathlib.Path("docs/mediciones/perf/estadisticas.json").read_text(encoding="utf-8"))
g = est["global"]
print("Serie K1 - backend local (n=3 corridas, 50 VUs, 30 s, GET /api/conductores)")
for c in est["por_corrida"]:
    print(f"{c['corrida']}: media={c['media_ms']:.3f}ms p95={c['p95_ms']:.2f}ms errores={c['error_rate']:.3f}")
print(f"Global K1: media_medias={g['media_medias_ms']:.2f}ms DT={g['desviacion_tipica_ms']:.2f}ms IC95=[{g['ic95_inf_ms']:.2f}; {g['ic95_sup_ms']:.2f}] p95max={g['p95_max_ms']:.2f}ms checks={g['checks_totales']}/{g['checks_fallos']}")


Serie K1 - backend local (n=3 corridas, 50 VUs, 30 s, GET /api/conductores)
k01-run1: media=37.324ms p95=173.13ms errores=0.000
k02-run2: media=16.897ms p95=35.30ms errores=0.000
k03-run3: media=14.818ms p95=37.84ms errores=0.000
Global K1: media_medias=23.01ms DT=12.44ms IC95=[-7.88; 53.91] p95max=173.13ms checks=8930/0


In [2]:
import json, pathlib
for k in ["k04", "k05", "k06", "k07", "k08"]:
    run = json.loads(pathlib.Path(f"docs/mediciones/perf/{k}-run1.json").read_text(encoding="utf-8"))
    cold = json.loads(pathlib.Path(f"docs/mediciones/perf/{k}-cold.json").read_text(encoding="utf-8"))
    d = run["metrics"]["http_req_duration"]
    ok = run["root_group"]["checks"]["status es 200"]
    print(f"{k.upper()} caliente: avg={d['avg']:.2f}ms med={d['med']:.2f}ms p95={d['p(95)']:.2f}ms status200={ok['passes']}/{ok['passes'] + ok['fails']}")
    print(f"{k.upper()} frio: primer GET={cold['metrics']['duracion_frio']['avg']:.2f}ms")


K04 caliente: avg=6294.89ms med=5889.13ms p95=11436.73ms status200=226/226
K04 frio: primer GET=156.52ms
K05 caliente: avg=4387.30ms med=3997.09ms p95=8411.26ms status200=297/297
K05 frio: primer GET=179.21ms
K06 caliente: avg=3771.27ms med=3654.86ms p95=7086.87ms status200=335/335
K06 frio: primer GET=207.89ms
K07 caliente: avg=3084.82ms med=2902.73ms p95=5356.12ms status200=392/392
K07 frio: primer GET=519.45ms
K08 caliente: avg=2245.02ms med=2099.45ms p95=4276.04ms status200=488/488
K08 frio: primer GET=224.42ms


## Figuras (se renderizan al abrir, sin re-ejecutar)
![Media con IC95](../docs/mediciones/perf/figuras/fig-media-ic95.png)
![p95 por corrida](../docs/mediciones/perf/figuras/fig-p95-por-corrida.png)
Generadas por `scripts/gen-figuras.py` (paleta Okabe-Ito). Ver también `fig-error-rate.png` y `fig-percentiles-corridas.png` en `docs/mediciones/perf/figuras/`.
## Lectura
K1 cumple el umbral p95 < 200 ms (p95 máx 173.13 ms, 0 % errores). En Render (plan gratuito, 0,1 vCPU) el p95 va de 4,3 s a 11,4 s: no cumple por throttling de CPU, no por regresión del código (contraste frío vs caliente p = 0,009 en `ANALISIS-k6.md`).